# Modul 7: Recurrent Neural Network

**Nama:** ISI NAMA  
**NIM:** ISI NIM  
**Kelas:** ISI KELAS  
**Tanggal:** YYYY-MM-DD  

Simpan berkas ini sebagai `M07_NIM.ipynb` sebelum mulai mengerjakan.

## Petunjuk

1. Ganti seluruh penanda `TODO`. Jangan menghapus sel pemeriksaan.
2. Vocabulary **wajib** dibangun dari subset latih saja.
3. Simpan panjang asli setiap contoh **sebelum** pembantalan.
4. Norma gradien dicatat **sebelum** pemotongan.
5. Protokol tetap: $12\,000$/$3\,000$, `MAKS=60`, Adam $10^{-3}$, batch $64$, $5$ epoch ($940$ update).
6. Luaran: `M07_NIM.ipynb`, `M07_NIM.pdf`, `M07_NIM_metrics.csv`.

In [ ]:
import platform
import random
import re
import time
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import DataLoader, TensorDataset

NIM = 'TODO'                 # contoh: '120450123'
SEED = int(str(NIM)[-4:]) if str(NIM).isdigit() else 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
pd.set_option('display.precision', 4)
print({'torch': torch.__version__, 'device': str(DEVICE), 'seed': SEED})

## A. Pre-lab - bagian dari 20 poin

1. **Mengapa vocabulary harus dibangun hanya dari data latih:** TODO
2. **Apa yang terjadi bila representasi diambil dari langkah terakhir pada batch berbantalan:** TODO
3. **Mengapa jumlah parameter RNN tidak bergantung panjang deret:** TODO
4. **Beda peran `<pad>` dan `<unk>`:** TODO

## B. Pipeline teks - bagian dari 20 poin

In [ ]:
ROOT = Path('../../data/raw/ag_news')
if not ROOT.exists():
    ROOT = Path('data/raw/ag_news')

kolom = ['label', 'judul', 'ringkasan']
tr = pd.read_csv(ROOT / 'train.csv', names=kolom, header=None)
te = pd.read_csv(ROOT / 'test.csv', names=kolom, header=None)
if not str(tr.iloc[0]['label']).strip().isdigit():
    tr, te = tr.iloc[1:].reset_index(drop=True), te.iloc[1:].reset_index(drop=True)

for df in (tr, te):
    df['teks'] = (df['judul'].astype(str) + ' ' + df['ringkasan'].astype(str))
    df['y'] = df['label'].astype(int) - 1

# TODO 1: split terstratifikasi 12.000 latih dan 3.000 validasi (random_state=SEED).
idx_latih, idx_val = ...

teks_latih, teks_val = tr['teks'].values[idx_latih], tr['teks'].values[idx_val]
y_latih, y_val = tr['y'].values[idx_latih], tr['y'].values[idx_val]

POLA = re.compile(r"[a-z0-9']+")
tokenisasi = lambda s: POLA.findall(s.lower())

# TODO 2: bangun vocabulary DARI teks_latih SAJA dengan min_freq=2.
#         Urutan: ['<pad>', '<unk>'] lalu token yang lolos ambang.
MIN_FREQ = 2
kosakata = ...
stoi = {w: i for i, w in enumerate(kosakata)}
V = len(kosakata)

# TODO 3: hitung persentase token validasi yang menjadi <unk>.
persen_unk = ...
print(f'vocabulary {V:,} | <unk> di validasi {persen_unk:.2f}%')

assert kosakata[0] == '<pad>' and kosakata[1] == '<unk>', 'urutan token khusus salah'
assert sorted(np.unique(y_latih)) == [0, 1, 2, 3], 'label harus 0..3'

In [ ]:
MAKS = 60

def ke_indeks(daftar_teks):
    """TODO 4: kembalikan (X, panjang).

    X berbentuk (N, MAKS) berisi indeks, dibantali dengan 0.
    panjang berisi jumlah token ASLI sebelum pembantalan (minimal 1).
    Token di luar vocabulary dipetakan ke 1 (<unk>); potong pada MAKS.
    """
    raise NotImplementedError

X_latih, L_latih = ke_indeks(teks_latih)
X_val, L_val = ke_indeks(teks_val)
ds_latih = TensorDataset(X_latih, L_latih, torch.tensor(y_latih))
ds_val = TensorDataset(X_val, L_val, torch.tensor(y_val))

print('X:', tuple(X_latih.shape), '| panjang median:', int(L_latih.median()))
assert X_latih.shape == (12_000, MAKS)
assert L_latih.min() >= 1 and L_latih.max() <= MAKS
assert (X_latih[0, L_latih[0]:] == 0).all(), 'sisa baris harus berisi <pad>'
print('pipeline teks sesuai protokol')

## C. Model dan bentuk tensor - 15 poin

In [ ]:
EMB, HID, KELAS = 100, 128, 4

class ModelRNN(nn.Module):
    def __init__(self, pakai_packing=True, hidden=HID):
        super().__init__()
        seed_everything(SEED)
        # TODO 5: Embedding(V, EMB, padding_idx=0), nn.RNN(EMB, hidden,
        #         batch_first=True), dan Linear(hidden, KELAS).
        raise NotImplementedError

    def forward(self, X, panjang):
        """TODO 6: bila pakai_packing, ambil h_n hasil pack_padded_sequence;
        bila tidak, ambil H[:, -1, :]. Kembalikan logit."""
        raise NotImplementedError

class RerataEmbedding(nn.Module):
    """TODO 7: pembanding tanpa RNN — rata-ratakan embedding sepanjang token
    ASLI (bagi dengan panjang, bukan dengan MAKS), lalu satu Linear."""
    def __init__(self):
        super().__init__()
        seed_everything(SEED)
        raise NotImplementedError

    def forward(self, X, panjang):
        raise NotImplementedError

m = ModelRNN()
xb, lb, yb = ds_latih[0:4]
with torch.no_grad():
    E = m.emb(xb); H, h_n = m.rnn(E)
print('X:', tuple(xb.shape), '| E:', tuple(E.shape),
      '| H:', tuple(H.shape), '| h_n:', tuple(h_n.shape))

bagian = {'embedding': sum(p.numel() for p in m.emb.parameters()),
          'rnn': sum(p.numel() for p in m.rnn.parameters()),
          'kepala': sum(p.numel() for p in m.kepala.parameters())}
tot = sum(bagian.values())
for k, v in bagian.items():
    print(f'  {k:>9}: {v:>10,} ({100*v/tot:5.1f}%)')
assert bagian['rnn'] == EMB * HID + HID * HID + 2 * HID, 'parameter RNN tidak sesuai rumus'

**Letak parameter.** Berapa persen parameter ada di tabel embedding, dan apa yang mengubah angka itu? TODO

## D. Membuktikan pengaruh bantalan - 20 poin

Inti modul. Bandingkan dua cara mengambil representasi akhir pada minibatch berisi contoh pendek dan panjang.

In [ ]:
# TODO 8: ambil satu contoh terpendek dan satu terpanjang dari validasi,
#         susun jadi satu batch, lalu hitung representasi akhir dengan DUA cara
#         memakai BOBOT YANG SAMA (salin state_dict).
#         Laporkan selisih maksimum untuk masing-masing contoh.
raise NotImplementedError

In [ ]:
# TODO 9: tambahkan 20 kolom <pad> di kanan batch tadi, hitung ulang keduanya,
#         lalu laporkan seberapa besar masing-masing berubah.
raise NotImplementedError

**Bukti angka.** Isi dengan hasil Anda:

- Selisih `h_n` lawan `H[:, -1]` pada contoh pendek: TODO
- Perubahan `h_n` setelah bantalan ditambah: TODO
- Perubahan `H[:, -1]` setelah bantalan ditambah: TODO
- Mengapa hanya salah satu yang tidak berubah? TODO

**Checkpoint menit ke-90.** Tunjukkan kepada asisten: ukuran vocabulary dan persentase `<unk>`, keempat bentuk tensor, serta bukti angka di atas.

## E. Empat run dan norma gradien - 25 poin

In [ ]:
BATCH, EPOCH = 64, 5

def loader(ds, batch, acak):
    g = torch.Generator().manual_seed(SEED)
    return DataLoader(ds, batch_size=batch, shuffle=acak, generator=g if acak else None)

@torch.no_grad()
def evaluasi(model, ds):
    """TODO 10: kembalikan (loss rata-rata, akurasi). Jangan lupa model.eval()."""
    raise NotImplementedError

def jalankan(model, label, clip=None, epoch=EPOCH):
    """TODO 11: satu fungsi pelatihan untuk SELURUH run.

    - Adam lr=1e-3, CrossEntropyLoss;
    - catat norma gradien SEBELUM pemotongan pada setiap update;
    - bila clip diberikan, panggil clip_grad_norm_ setelah pencatatan;
    - kembalikan (model, riwayat, catatan) dengan catatan memuat run_id, seed,
      model, vocab, panjang_maks, hidden, parameter, n_update, clip,
      train_loss, val_loss, val_acc, grad_norm_mean, detik_per_epoch.
    """
    raise NotImplementedError

hasil, kurva, simpan = [], {}, {}
for model, label, clip in [(RerataEmbedding(), 'rerata-embedding', None),
                           (ModelRNN(False), 'rnn-tanpa-packing', None),
                           (ModelRNN(True), 'rnn-packing', None),
                           (ModelRNN(True), 'rnn-packing-clip', 1.0)]:
    m_, r, catatan = jalankan(model, label, clip)
    hasil.append(catatan); kurva[label] = r; simpan[label] = m_

print(pd.DataFrame(hasil)[['run_id', 'parameter', 'n_update', 'val_loss',
                           'val_acc', 'grad_norm_mean',
                           'detik_per_epoch']].to_string(index=False))
assert len(hasil) == 4 and len({c['n_update'] for c in hasil}) == 1, \
    'keempat run harus memakai anggaran update yang sama'

In [ ]:
# TODO 12: dua panel grafik — (kiri) validation accuracy keempat run,
#          (kanan) norma gradien dengan dan tanpa pemotongan (skala log).
raise NotImplementedError

## F. Run tambahan - bagian dari 25 poin

In [ ]:
# TODO 13: dua run panjang maksimum (MAKS=30 dan MAKS=60) dan
#          dua run ukuran keadaan (hidden=64 dan 128), seluruhnya pada
#          konfigurasi terbaik. Tambahkan ke daftar `hasil`.
#          Petunjuk: untuk MAKS=30, bangun ulang X dan L dengan ke_indeks.
raise NotImplementedError

tabel = pd.DataFrame(hasil)
print(tabel[['run_id', 'panjang_maks', 'hidden', 'val_acc',
             'detik_per_epoch']].to_string(index=False))
assert len(tabel) >= 7, 'minimal empat run inti ditambah run tambahan'

## G. Akurasi menurut panjang teks - bagian dari 10 poin

In [ ]:
# TODO 14: kelompokkan validasi menjadi <20, 20-40, dan >40 token,
#          lalu laporkan akurasi dan jumlah contoh tiap golongan
#          untuk model terbaik.
raise NotImplementedError

In [ ]:
# TODO 15: simpan seluruh run ke metrics.csv.
tabel.insert(0, 'module', 'M07')
tabel.insert(1, 'student_id', NIM)
tabel.to_csv(f'M07_{NIM}_metrics.csv', index=False)
print(f'{len(tabel)} baris tersimpan')

## H. Pertanyaan analisis - bagian dari 10 poin

1. Berapa selisih akurasi RNN dengan dan tanpa pengemasan, dan mengapa kesalahan itu tidak melempar galat? TODO
2. Apakah RNN mengalahkan pembanding rerata embedding? Bila selisihnya kecil, apa artinya? TODO
3. Berapa persen parameter ada di tabel embedding, dan apa yang mengubah angka itu? TODO
4. Bagaimana perbandingan norma gradien dengan dan tanpa pemotongan, dan apakah akurasinya berubah? TODO
5. Golongan panjang mana yang paling banyak salah, dan apa dugaan penyebabnya? TODO

## Checklist sebelum mengumpulkan

- [ ] Identitas, seed, device, dan ukuran vocabulary tercantum.
- [ ] Vocabulary dibangun dari subset latih saja.
- [ ] Panjang asli disimpan sebelum pembantalan.
- [ ] Selisih `H[:, -1]` lawan `h_n` dilaporkan sebagai angka.
- [ ] Norma gradien dicatat sebelum pemotongan.
- [ ] Pembanding rerata embedding disertakan.
- [ ] `metrics.csv` memuat seluruh run.
- [ ] Notebook lolos *Restart Kernel and Run All*.